# Clase 9 - Agente con herramientas y Firestore/Firebase

<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Google Colab"/></a>

> Para usarlo en Google Colab: sube este notebook a Colab o guárdalo en Drive y ábrelo desde `Archivo > Abrir notebook`.

**Objetivos.** Al terminar deberías poder:

- Crear paso a paso un proyecto Firebase, una base Firestore y una conexión desde Colab/Python.
- Definir un esquema simple de datos para un agente.
- Implementar herramientas de lectura/escritura y entender cómo extenderlas a email, Slack, CRM u otras APIs.
- Construir un agente con ciclo percepción, planificación, acción, observación y memoria.
- Registrar trazas de acciones del agente.

## Arquitectura que construiremos

```text
Usuario
  |
  v
Agente de soporte
  |       
  |-- interpreta solicitud
  |-- elige herramienta
  |-- valida parámetros
  |-- ejecuta acción
  |-- registra memoria
  v
Respuesta final

Herramientas:
- buscar_cliente
- crear_ticket
- listar_tickets
- actualizar_estado_ticket
- resumir_tickets
```

## Crear y preparar Firebase/Firestore

Antes de conectar el agente a una base real, prepararémos un proyecto de Firebase. Este paso se hace una sola vez por proyecto.

### i. Crear proyecto

1. Abre https://console.firebase.google.com/.
2. Selecciona **Agregar proyecto**.
3. Define un nombre, por ejemplo `agente-soporte-demo`.
4. Google Analytics es opcional para este laboratorio.
5. Espera a que el proyecto quede creado.

### ii. Crear base Firestore

1. En el panel izquierdo ve a **Bases de datos y almacenamiento > Firestore Database**.
2. Haz clic en **Crear base de datos**.
4. Usar **Modo de producción.**.

Firestore organiza datos como:

```text
coleccion/documento/campos
```

En este laboratorio usaremos:

```text
clientes/{cliente_id}
tickets/{ticket_id}
memoria_agente/{evento_id}
```

### iii. Crear credencial de servidor

1. En Firebase Console, abre **Configuración > General**.
2. Ve a la pestaña **Cuentas de Servicio**.
3. Presiona **Generar nueva clave privada**.
4. Descarga el JSON.
5. Guarda ese archivo fuera del repositorio.

Importante: ese JSON permite acceso privilegiado desde servidor. No lo pegues en el notebook, no lo subas a GitHub y no lo compartas.

In [ ]:
#%pip install -q -U firebase-admin pandas pydantic requests openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import re
import time
import uuid
from collections import Counter
from copy import deepcopy
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any, Callable, Dict, List, Optional

import pandas as pd
from pydantic import BaseModel, Field, ValidationError

## Paso 1: configurar credenciales en Colab o local

### Opción A: archivo JSON subido a Colab

```python
from google.colab import files
uploaded = files.upload()
RUTA_SERVICE_ACCOUNT = "/content/nombre-del-archivo.json"
```

Ventaja: simple para clase.  
Cuidado: el archivo queda disponible durante la sesión de Colab; elimínalo al terminar.

### Opción B: variable de entorno `GOOGLE_APPLICATION_CREDENTIALS`

En local:

```bash
export GOOGLE_APPLICATION_CREDENTIALS="/ruta/serviceAccountKey.json"
```

En el notebook puedes leer esa ruta con `os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")`.

In [3]:
RUTA_SERVICE_ACCOUNT = "agente-soporte-aab7c-firebase-adminsdk-fbsvc-1f5c613fe9.json"

# Opcional si usas Application Default Credentials y necesitas fijar proyecto.
FIREBASE_PROJECT_ID = "agente-soporte-aab7c"

# Nombre interno de la app Firebase dentro de esta sesión Python.
NOMBRE_APP_FIREBASE = "agente-soporte"

print("Ruta service account:", RUTA_SERVICE_ACCOUNT)
print("FIREBASE_PROJECT_ID:", FIREBASE_PROJECT_ID)

Ruta service account: agente-soporte-aab7c-firebase-adminsdk-fbsvc-1f5c613fe9.json
FIREBASE_PROJECT_ID: agente-soporte-aab7c


## Paso 1.1: subir credencial desde Colab, si corresponde

Activa la siguiente celda solo si ejecutarás el Notebook desde Google Colab.

In [ ]:
# from google.colab import files
# uploaded = files.upload()
#
# # Si subiste un solo archivo JSON, puedes tomar su nombre así:
# nombre_archivo = next(iter(uploaded.keys()))
# RUTA_SERVICE_ACCOUNT = f"/content/{nombre_archivo}"
# USAR_FIRESTORE_REAL = True
#
# print("Credencial cargada en:", RUTA_SERVICE_ACCOUNT)
# print("Existe archivo:", os.path.exists(RUTA_SERVICE_ACCOUNT))

## Paso 2: datos y esquema

Usaremos tres colecciones:

| Colección | Descripción |
|---|---|
| `clientes` | Datos básicos del cliente |
| `tickets` | Casos de soporte |
| `memoria_agente` | Trazas y acciones del agente |

In [8]:
CLIENTES_SEMILLA = [
    {"id": "cli_001", "nombre": "Ana Torres", "email": "ana@empresa.cl", "plan": "premium", "riesgo": "medio"},
    {"id": "cli_002", "nombre": "Bruno Díaz", "email": "bruno@empresa.cl", "plan": "standard", "riesgo": "bajo"},
    {"id": "cli_003", "nombre": "Carla Soto", "email": "carla@empresa.cl", "plan": "premium", "riesgo": "alto"},
]

TICKETS_SEMILLA = [
    {"id": "tic_001", "cliente_id": "cli_001", "asunto": "No puede iniciar sesión", "prioridad": "alta", "estado": "abierto"},
    {"id": "tic_002", "cliente_id": "cli_002", "asunto": "Consulta sobre facturación", "prioridad": "media", "estado": "abierto"},
    {"id": "tic_003", "cliente_id": "cli_003", "asunto": "Error intermitente en dashboard", "prioridad": "alta", "estado": "en_progreso"},
]

ESTADOS_VALIDOS = {"abierto", "en_progreso", "cerrado"}
PRIORIDADES_VALIDAS = {"baja", "media", "alta", "critica"}

pd.DataFrame(CLIENTES_SEMILLA), pd.DataFrame(TICKETS_SEMILLA)

(        id      nombre             email      plan riesgo
 0  cli_001  Ana Torres    ana@empresa.cl   premium  medio
 1  cli_002  Bruno Díaz  bruno@empresa.cl  standard   bajo
 2  cli_003  Carla Soto  carla@empresa.cl   premium   alto,
         id cliente_id                           asunto prioridad       estado
 0  tic_001    cli_001          No puede iniciar sesión      alta      abierto
 1  tic_002    cli_002       Consulta sobre facturación     media      abierto
 2  tic_003    cli_003  Error intermitente en dashboard      alta  en_progreso)

## Paso 3: repositorio Firestore

In [9]:
class RepositorioFirestore:
    def __init__(self, db):
        self.db = db

    def _coleccion(self, nombre):
        return self.db.collection(nombre)

    def sembrar(self, clientes, tickets):
        for cliente in clientes:
            self._coleccion("clientes").document(cliente["id"]).set(deepcopy(cliente))
        for ticket in tickets:
            doc = deepcopy(ticket)
            doc.setdefault("creado_en", ahora_iso())
            doc.setdefault("actualizado_en", ahora_iso())
            self._coleccion("tickets").document(doc["id"]).set(doc)

    def listar_clientes(self):
        return [doc.to_dict() for doc in self._coleccion("clientes").stream()]

    def buscar_cliente(self, texto):
        texto = texto.lower().strip()
        resultados = []
        for cliente in self.listar_clientes():
            if texto in cliente.get("nombre", "").lower() or texto in cliente.get("email", "").lower() or texto == cliente.get("id", "").lower():
                resultados.append(cliente)
        return resultados

    def obtener_cliente(self, cliente_id):
        doc = self._coleccion("clientes").document(cliente_id).get()
        return doc.to_dict() if doc.exists else None

    def crear_ticket(self, cliente_id, asunto, prioridad):
        existentes = list(self._coleccion("tickets").stream())
        ticket_id = f"tic_{len(existentes) + 1:03d}"
        doc = {
            "id": ticket_id,
            "cliente_id": cliente_id,
            "asunto": asunto,
            "prioridad": prioridad,
            "estado": "abierto",
            "creado_en": ahora_iso(),
            "actualizado_en": ahora_iso(),
        }
        self._coleccion("tickets").document(ticket_id).set(doc)
        return doc

    def listar_tickets(self, estado=None, prioridad=None, cliente_id=None):
        docs = [doc.to_dict() for doc in self._coleccion("tickets").stream()]
        if estado:
            docs = [d for d in docs if d.get("estado") == estado]
        if prioridad:
            docs = [d for d in docs if d.get("prioridad") == prioridad]
        if cliente_id:
            docs = [d for d in docs if d.get("cliente_id") == cliente_id]
        return docs

    def actualizar_estado_ticket(self, ticket_id, estado):
        ref = self._coleccion("tickets").document(ticket_id)
        doc = ref.get()
        if not doc.exists:
            return None
        ref.update({"estado": estado, "actualizado_en": ahora_iso()})
        actualizado = ref.get()
        return actualizado.to_dict()

    def registrar_memoria(self, evento):
        evento_id = evento.get("id", f"mem_{uuid.uuid4().hex[:8]}")
        evento["id"] = evento_id
        evento.setdefault("creado_en", ahora_iso())
        self._coleccion("memoria_agente").document(evento_id).set(deepcopy(evento))
        return evento

    def listar_memoria(self, limite=20):
        docs = [doc.to_dict() for doc in self._coleccion("memoria_agente").stream()]
        docs = sorted(docs, key=lambda x: x.get("creado_en", ""), reverse=True)
        return docs[:limite]

def ahora_iso():
    return datetime.now(timezone.utc).isoformat()

## Paso 4: conectar al repositorio

La función siguiente elige Firestore real o repositorio simulado. Si Firestore falla, no intenta esconder el problema: muestra el error y usa modo simulado para continuar el laboratorio.

In [11]:
def conectar_repositorio():
    try:
        import firebase_admin
        from firebase_admin import credentials, firestore

        # Reutiliza la app si ya fue inicializada en esta sesión.
        if NOMBRE_APP_FIREBASE in firebase_admin._apps:
            app = firebase_admin.get_app(NOMBRE_APP_FIREBASE)
        else:
            opciones = {}
            if FIREBASE_PROJECT_ID:
                opciones["projectId"] = FIREBASE_PROJECT_ID

            if os.path.exists(RUTA_SERVICE_ACCOUNT):
                print("Inicializando Firebase con service account JSON.")
                cred = credentials.Certificate(RUTA_SERVICE_ACCOUNT)
                app = firebase_admin.initialize_app(cred, options=opciones, name=NOMBRE_APP_FIREBASE)
            else:
                print("No se encontró JSON. Intentando Application Default Credentials.")
                cred = credentials.ApplicationDefault()
                app = firebase_admin.initialize_app(cred, options=opciones, name=NOMBRE_APP_FIREBASE)

        db = firestore.client(app=app)

        # Prueba liviana: listar una cantidad pequeña de documentos.
        _ = list(db.collection("clientes").limit(1).stream())

        repo = RepositorioFirestore(db)
        print("Conectado a Firestore.")
        return repo, "firestore"
    except Exception as error:
        print("No se pudo conectar a Firestore real.")
        print("Motivo:", repr(error))

repo, modo_repo = conectar_repositorio()

Conectado a Firestore.


## Paso 4.1: sembrar datos demo en Firestore real

Si conectaste Firestore real, esta celda puede escribir documentos demo en las colecciones `clientes` y `tickets`.

Por seguridad viene desactivada. Activa `SEMBRAR_DATOS_DEMO = True` solo cuando estés usando un proyecto de práctica.

In [12]:
repo.sembrar(CLIENTES_SEMILLA, TICKETS_SEMILLA)
print("Datos demo escritos en Firestore.")

Datos demo escritos en Firestore.


## Paso 4.2: verificar en Firebase Console

1. Vuelve a Firebase Console.
2. Accede a **Firestore**.
3. Verifica que existan las colecciones `clientes`, `tickets`.
4. Revisa que los documentos tengan campos como `id`, `cliente_id`, `estado`, `prioridad`, `creado_en`.

## Paso 5: verificar datos

Antes de construir el agente, verificamos que podemos leer clientes y tickets.

In [13]:
clientes = repo.listar_clientes()
tickets = repo.listar_tickets()

print("Clientes:")
display(pd.DataFrame(clientes))
print("Tickets:")
display(pd.DataFrame(tickets))

Clientes:


,riesgo,nombre,plan,id,email
0,medio,Ana Torres,premium,cli_001,ana@empresa.cl
1,bajo,Bruno Díaz,standard,cli_002,bruno@empresa.cl
2,alto,Carla Soto,premium,cli_003,carla@empresa.cl


Tickets:


,actualizado_en,id,prioridad,cliente_id,estado,asunto,creado_en
0,2026-08-02T18:08:57.762455+00:00,tic_001,alta,cli_001,abierto,No puede iniciar sesión,2026-08-02T18:08:57.762437+00:00
1,2026-08-02T18:08:58.117541+00:00,tic_002,media,cli_002,abierto,Consulta sobre facturación,2026-08-02T18:08:58.117522+00:00
2,2026-08-02T18:08:58.738583+00:00,tic_003,alta,cli_003,en_progreso,Error intermitente en dashboard,2026-08-02T18:08:58.738564+00:00


## Paso 6: definir esquemas de herramientas

Las herramientas deben recibir entradas válidas. Usaremos Pydantic para validar parámetros antes de tocar la base.

In [14]:
class BuscarClienteInput(BaseModel):
    texto: str = Field(..., description="Nombre, email o ID del cliente")

class CrearTicketInput(BaseModel):
    cliente_id: str
    asunto: str
    prioridad: str

class ListarTicketsInput(BaseModel):
    estado: Optional[str] = None
    prioridad: Optional[str] = None
    cliente_id: Optional[str] = None

class ActualizarEstadoTicketInput(BaseModel):
    ticket_id: str
    estado: str
    confirmado: bool = False

class ToolResult(BaseModel):
    ok: bool
    mensaje: str
    datos: Any = None

## Paso 6.1: catálogo de herramientas del agente

Antes de implementar funciones, definimos el contrato de cada herramienta. Este contrato es lo que luego podriamos entregar a un LLM con tool calling o a un framework como LangChain/LangGraph.

| Herramienta | Lee/escribe | Parámetros | Confirmación |
|---|---|---|---|
| `buscar_cliente` | Lee `clientes` | `texto` | No |
| `crear_ticket` | Escribe `tickets` | `cliente_id`, `asunto`, `prioridad` | No |
| `listar_tickets` | Lee `tickets` | `estado`, `prioridad`, `cliente_id` | No |
| `actualizar_estado_ticket` | Actualiza `tickets` | `ticket_id`, `estado`, `confirmado` | Sí |
| `resumir_tickets` | Lee `tickets` | ninguno | No |

Principio de diseño: el agente no toca Firestore directamente; solo puede actuar mediante estas herramientas validadas.

In [15]:
CATALOGO_HERRAMIENTAS = [
    {
        "nombre": "buscar_cliente",
        "descripcion": "Busca clientes por nombre, email o ID.",
        "lectura": ["clientes"],
        "escritura": [],
        "requiere_confirmacion": False,
    },
    {
        "nombre": "crear_ticket",
        "descripcion": "Crea un ticket de soporte para un cliente existente.",
        "lectura": ["clientes"],
        "escritura": ["tickets"],
        "requiere_confirmacion": False,
    },
    {
        "nombre": "listar_tickets",
        "descripcion": "Lista tickets filtrando por estado, prioridad o cliente.",
        "lectura": ["tickets"],
        "escritura": [],
        "requiere_confirmacion": False,
    },
    {
        "nombre": "actualizar_estado_ticket",
        "descripcion": "Cambia el estado de un ticket existente.",
        "lectura": ["tickets"],
        "escritura": ["tickets"],
        "requiere_confirmacion": True,
    },
    {
        "nombre": "resumir_tickets",
        "descripcion": "Resume tickets por estado y prioridad.",
        "lectura": ["tickets"],
        "escritura": [],
        "requiere_confirmacion": False,
    },
]

pd.DataFrame(CATALOGO_HERRAMIENTAS)

,nombre,descripcion,lectura,escritura,requiere_confirmacion
0,buscar_cliente,"Busca clientes por nombre, email o ID.",[clientes],[],False
1,crear_ticket,Crea un ticket de soporte para un cliente exis...,[clientes],[tickets],False
2,listar_tickets,"Lista tickets filtrando por estado, prioridad ...",[tickets],[],False
3,actualizar_estado_ticket,Cambia el estado de un ticket existente.,[tickets],[tickets],True
4,resumir_tickets,Resume tickets por estado y prioridad.,[tickets],[],False


## Paso 7: implementar herramientas

Estas funciones son los actuadores del agente. Observa que validan entradas y devuelven resultados estructurados.

In [16]:
def herramienta_buscar_cliente(texto: str) -> ToolResult:
    try:
        # Crear el objeto de validación para verificar estructura esperada.
        args = BuscarClienteInput(texto=texto)
        # Llamar a la herramienta 
        resultados = repo.buscar_cliente(args.texto)
        if not resultados:
            return ToolResult(ok=False, mensaje="No se encontraron clientes.", datos=[])
        return ToolResult(ok=True, mensaje=f"Se encontraron {len(resultados)} cliente(s).", datos=resultados)
    except ValidationError as error:
        return ToolResult(ok=False, mensaje="Entrada inválida para buscar_cliente.", datos=str(error))


def herramienta_crear_ticket(cliente_id: str, asunto: str, prioridad: str) -> ToolResult:
    try:
        args = CrearTicketInput(cliente_id=cliente_id, asunto=asunto, prioridad=prioridad.lower())
        if args.prioridad not in PRIORIDADES_VALIDAS:
            return ToolResult(ok=False, mensaje=f"Prioridad inválida: {args.prioridad}.", datos={"validas": sorted(PRIORIDADES_VALIDAS)})
        cliente = repo.obtener_cliente(args.cliente_id)
        if not cliente:
            return ToolResult(ok=False, mensaje="Cliente no existe.", datos={"cliente_id": args.cliente_id})
        ticket = repo.crear_ticket(args.cliente_id, args.asunto, args.prioridad)
        return ToolResult(ok=True, mensaje="Ticket creado correctamente.", datos=ticket)
    except ValidationError as error:
        return ToolResult(ok=False, mensaje="Entrada inválida para crear_ticket.", datos=str(error))


def herramienta_listar_tickets(estado=None, prioridad=None, cliente_id=None) -> ToolResult:
    try:
        args = ListarTicketsInput(estado=estado, prioridad=prioridad, cliente_id=cliente_id)
        if args.estado and args.estado not in ESTADOS_VALIDOS:
            return ToolResult(ok=False, mensaje=f"Estado inválido: {args.estado}.", datos={"validos": sorted(ESTADOS_VALIDOS)})
        if args.prioridad and args.prioridad not in PRIORIDADES_VALIDAS:
            return ToolResult(ok=False, mensaje=f"Prioridad inválida: {args.prioridad}.", datos={"validas": sorted(PRIORIDADES_VALIDAS)})
        tickets = repo.listar_tickets(estado=args.estado, prioridad=args.prioridad, cliente_id=args.cliente_id)
        return ToolResult(ok=True, mensaje=f"Se encontraron {len(tickets)} ticket(s).", datos=tickets)
    except ValidationError as error:
        return ToolResult(ok=False, mensaje="Entrada inválida para listar_tickets.", datos=str(error))


def herramienta_actualizar_estado_ticket(ticket_id: str, estado: str, confirmado: bool = False) -> ToolResult:
    try:
        args = ActualizarEstadoTicketInput(ticket_id=ticket_id, estado=estado.lower(), confirmado=confirmado)
        if args.estado not in ESTADOS_VALIDOS:
            return ToolResult(ok=False, mensaje=f"Estado inválido: {args.estado}.", datos={"validos": sorted(ESTADOS_VALIDOS)})
        if not args.confirmado:
            return ToolResult(ok=False, mensaje="Se requiere confirmación humana para cambiar estado.", datos={"requiere_confirmacion": True})
        ticket = repo.actualizar_estado_ticket(args.ticket_id, args.estado)
        if not ticket:
            return ToolResult(ok=False, mensaje="Ticket no encontrado.", datos={"ticket_id": args.ticket_id})
        return ToolResult(ok=True, mensaje="Estado actualizado correctamente.", datos=ticket)
    except ValidationError as error:
        return ToolResult(ok=False, mensaje="Entrada inválida para actualizar_estado_ticket.", datos=str(error))


def herramienta_resumir_tickets() -> ToolResult:
    tickets = repo.listar_tickets()
    resumen = Counter((t.get("estado"), t.get("prioridad")) for t in tickets)
    filas = [
        {"estado": estado, "prioridad": prioridad, "cantidad": cantidad}
        for (estado, prioridad), cantidad in resumen.items()
    ]
    return ToolResult(ok=True, mensaje="Resumen generado.", datos=filas)

In [17]:
# Prueba directa de herramientas.
print(herramienta_buscar_cliente("Ana").model_dump())
print(herramienta_listar_tickets(estado="abierto").model_dump())
print(herramienta_resumir_tickets().model_dump())

{'ok': True, 'mensaje': 'Se encontraron 1 cliente(s).', 'datos': [{'riesgo': 'medio', 'nombre': 'Ana Torres', 'plan': 'premium', 'id': 'cli_001', 'email': 'ana@empresa.cl'}]}
{'ok': True, 'mensaje': 'Se encontraron 2 ticket(s).', 'datos': [{'actualizado_en': '2026-08-02T18:08:57.762455+00:00', 'id': 'tic_001', 'prioridad': 'alta', 'cliente_id': 'cli_001', 'estado': 'abierto', 'asunto': 'No puede iniciar sesión', 'creado_en': '2026-08-02T18:08:57.762437+00:00'}, {'actualizado_en': '2026-08-02T18:08:58.117541+00:00', 'id': 'tic_002', 'prioridad': 'media', 'cliente_id': 'cli_002', 'estado': 'abierto', 'asunto': 'Consulta sobre facturación', 'creado_en': '2026-08-02T18:08:58.117522+00:00'}]}
{'ok': True, 'mensaje': 'Resumen generado.', 'datos': [{'estado': 'abierto', 'prioridad': 'alta', 'cantidad': 1}, {'estado': 'abierto', 'prioridad': 'media', 'cantidad': 1}, {'estado': 'en_progreso', 'prioridad': 'alta', 'cantidad': 1}]}


## Paso 8: registrar memoria del agente

Cada interacción debe dejar una traza: solicitud, plan, acciones, observaciones y respuesta. Esto permite depurar y auditar.

In [18]:
def registrar_evento(tipo: str, contenido: Dict[str, Any]) -> Dict[str, Any]:
    evento = {
        "tipo": tipo,
        "contenido": contenido,
        "creado_en": ahora_iso(),
        "modo_repo": modo_repo,
    }
    return repo.registrar_memoria(evento)

registrar_evento("prueba", {"mensaje": "Memoria inicial del agente"})
pd.DataFrame(repo.listar_memoria())

,tipo,id,creado_en,modo_repo,contenido
0,prueba,mem_154da911,2026-08-02T18:30:34.880429+00:00,firestore,{'mensaje': 'Memoria inicial del agente'}


## Paso 9: definir acciones y planificador

En producción, el planificador puede ser un LLM con tool calling. En este laboratorio implementamos un planificador transparente y verificable para concentrarnos en el ciclo agente-herramienta-base.

El planificador convierte lenguaje natural en acciones permitidas.

In [23]:
@dataclass
class Accion:
    nombre: str
    argumentos: Dict[str, Any]
    requiere_confirmacion: bool = False


def normalizar_texto(texto: str) -> str:
    return texto.lower().strip()


def detectar_prioridad(texto: str) -> Optional[str]:
    texto = normalizar_texto(texto)
    for prioridad in ["critica", "alta", "media", "baja"]:
        if prioridad in texto:
            return prioridad
    return None


def detectar_estado(texto: str) -> Optional[str]:
    texto = normalizar_texto(texto)
    equivalencias = {
        "abiertos": "abierto",
        "abierto": "abierto",
        "en progreso": "en_progreso",
        "en_progreso": "en_progreso",
        "cerrados": "cerrado",
        "cerrado": "cerrado",
        "cerrar": "cerrado",
        "cierra": "cerrado",
    }
    for clave, valor in equivalencias.items():
        if clave in texto:
            return valor
    return None


def detectar_ticket_id(texto: str) -> Optional[str]:
    m = re.search(r"tic_\d{3}", texto.lower())
    return m.group(0) if m else None


def extraer_asunto(texto: str) -> str:
    patrones = [r"asunto[:\-]\s*(.+)", r"por[:\-]\s*(.+)", r"sobre[:\-]\s*(.+)"]
    for patron in patrones:
        m = re.search(patron, texto, flags=re.IGNORECASE)
        if m:
            return m.group(1).strip().rstrip(".")
    return texto.strip()


def resolver_cliente_id(texto: str) -> Optional[str]:
    texto_norm = normalizar_texto(texto)
    for cliente in repo.listar_clientes():
        if cliente["id"].lower() in texto_norm or cliente["nombre"].lower() in texto_norm:
            return cliente["id"]
    return None


def planificar(mensaje_usuario: str) -> List[Accion]:
    texto = normalizar_texto(mensaje_usuario)

    if any(p in texto for p in ["resumen", "resume", "estado general", "dashboard"]):
        return [Accion("resumir_tickets", {})]

    if "buscar" in texto and "cliente" in texto:
        consulta = mensaje_usuario.split("cliente", 1)[-1].strip(" :.-") or mensaje_usuario
        return [Accion("buscar_cliente", {"texto": consulta})]

    if "crear" in texto and "ticket" in texto:
        cliente_id = resolver_cliente_id(mensaje_usuario)
        prioridad = detectar_prioridad(mensaje_usuario) or "media"
        asunto = extraer_asunto(mensaje_usuario)
        if not cliente_id:
            return [Accion("buscar_cliente", {"texto": mensaje_usuario})]
        return [Accion("crear_ticket", {"cliente_id": cliente_id, "asunto": asunto, "prioridad": prioridad})]

    if any(p in texto for p in ["listar", "lista", "muestra", "mostrar", "ver"]) and "ticket" in texto:
        return [Accion("listar_tickets", {
            "estado": detectar_estado(mensaje_usuario),
            "prioridad": detectar_prioridad(mensaje_usuario),
            "cliente_id": resolver_cliente_id(mensaje_usuario),
        })]

    if any(p in texto for p in ["cerrar", "cierra", "actualiza", "cambiar estado"]):
        ticket_id = detectar_ticket_id(mensaje_usuario)
        estado = detectar_estado(mensaje_usuario) or "cerrado"
        return [Accion("actualizar_estado_ticket", {"ticket_id": ticket_id, "estado": estado}, requiere_confirmacion=True)]

    return [Accion("ayuda", {"mensaje": mensaje_usuario})]

for consulta in [
    "Buscar cliente Ana",
    "Crear ticket para Ana Torres por error de login prioridad alta",
    "Lista tickets abiertos de prioridad alta",
    "Cierra tic_001",
    "Dame un resumen",
]:
    print(consulta, "->", planificar(consulta))

Buscar cliente Ana -> [Accion(nombre='buscar_cliente', argumentos={'texto': 'Ana'}, requiere_confirmacion=False)]
Crear ticket para Ana Torres por error de login prioridad alta -> [Accion(nombre='crear_ticket', argumentos={'cliente_id': 'cli_001', 'asunto': 'Crear ticket para Ana Torres por error de login prioridad alta', 'prioridad': 'alta'}, requiere_confirmacion=False)]
Lista tickets abiertos de prioridad alta -> [Accion(nombre='listar_tickets', argumentos={'estado': 'abierto', 'prioridad': 'alta', 'cliente_id': None}, requiere_confirmacion=False)]
Cierra tic_001 -> [Accion(nombre='actualizar_estado_ticket', argumentos={'ticket_id': 'tic_001', 'estado': 'cerrado'}, requiere_confirmacion=True)]
Dame un resumen -> [Accion(nombre='resumir_tickets', argumentos={}, requiere_confirmacion=False)]


## Paso 10: ejecutar acciones

El ejecutor toma el plan, llama herramientas y devuelve observaciones estructuradas.

In [29]:
HERRAMIENTAS: Dict[str, Callable[..., ToolResult]] = {
    "buscar_cliente": herramienta_buscar_cliente,
    "crear_ticket": herramienta_crear_ticket,
    "listar_tickets": herramienta_listar_tickets,
    "actualizar_estado_ticket": herramienta_actualizar_estado_ticket,
    "resumir_tickets": herramienta_resumir_tickets,
}


def ejecutar_accion(accion: Accion, confirmar: bool = False) -> ToolResult:
    if accion.nombre == "ayuda":
        return ToolResult(
            ok=False,
            mensaje="Puedo buscar clientes, crear tickets, listar tickets, cerrar tickets o resumir estado.",
            datos={"solicitud": accion.argumentos.get("mensaje")},
        )

    herramienta = HERRAMIENTAS.get(accion.nombre)
    if not herramienta:
        return ToolResult(ok=False, mensaje=f"Herramienta no permitida: {accion.nombre}", datos=None)

    argumentos = dict(accion.argumentos)
    if accion.requiere_confirmacion:
        argumentos["confirmado"] = confirmar
    return herramienta(**argumentos)


def formatear_respuesta(accion: Accion, resultado: ToolResult) -> str:
    if not resultado.ok:
        return f"No pude completar `{accion.nombre}`: {resultado.mensaje}. Datos: {resultado.datos}"

    if accion.nombre == "buscar_cliente":
        filas = [f"- {c['id']}: {c['nombre']} ({c['email']}), plan {c['plan']}, riesgo {c['riesgo']}" for c in resultado.datos]
        return "Clientes encontrados:\n" + "\n".join(filas)

    if accion.nombre == "crear_ticket":
        t = resultado.datos
        return f"Ticket creado: {t['id']} para {t['cliente_id']} | prioridad {t['prioridad']} | estado {t['estado']} | asunto: {t['asunto']}"

    if accion.nombre == "listar_tickets":
        if not resultado.datos:
            return "No hay tickets con esos filtros."
        filas = [f"- {t['id']} | cliente {t['cliente_id']} | {t['estado']} | {t['prioridad']} | {t['asunto']}" for t in resultado.datos]
        return "Tickets encontrados:\n" + "\n".join(filas)

    if accion.nombre == "actualizar_estado_ticket":
        t = resultado.datos
        return f"Ticket {t['id']} actualizado a estado `{t['estado']}`."

    if accion.nombre == "resumir_tickets":
        if not resultado.datos:
            return "No hay tickets para resumir."
        filas = [f"- {r['estado']} - {r['prioridad']}: {r['cantidad']}" for r in resultado.datos]
        return "Resumen de tickets:\n" + "\n".join(filas)

    return resultado.mensaje

## Paso 11: construir el agente

El agente integra percepción, planificación, acción, observación, memoria y respuesta.

In [25]:
class AgenteSoporte:
    def __init__(self, repo):
        self.repo = repo

    def responder(self, mensaje_usuario: str, confirmar: bool = False) -> Dict[str, Any]:
        inicio = time.time()
        plan = planificar(mensaje_usuario)
        trazas = []
        respuestas = []

        for accion in plan:
            resultado = ejecutar_accion(accion, confirmar=confirmar)
            respuesta = formatear_respuesta(accion, resultado)
            trazas.append({
                "accion": accion.nombre,
                "argumentos": accion.argumentos,
                "requiere_confirmacion": accion.requiere_confirmacion,
                "confirmado": confirmar,
                "resultado": resultado.model_dump(),
            })
            respuestas.append(respuesta)

        salida = "\n\n".join(respuestas)
        evento = registrar_evento("interaccion_agente", {
            "mensaje_usuario": mensaje_usuario,
            "plan": [a.__dict__ for a in plan],
            "trazas": trazas,
            "respuesta": salida,
            "segundos": round(time.time() - inicio, 3),
        })

        return {
            "respuesta": salida,
            "plan": [a.__dict__ for a in plan],
            "trazas": trazas,
            "memoria_id": evento["id"],
        }

agente = AgenteSoporte(repo)
print("Agente inicializado.")

Agente inicializado.


In [38]:
casos_prueba = [
    {"consulta": "Buscar cliente Carla", "accion_esperada": "buscar_cliente"},
    {"consulta": "Crear ticket para Bruno Díaz por duda de factura prioridad media", "accion_esperada": "crear_ticket"},
    {"consulta": "Muestra tickets abiertos", "accion_esperada": "listar_tickets"},
    {"consulta": "Dame un resumen de tickets", "accion_esperada": "resumir_tickets"},
    {"consulta": "Cierra tic_002", "accion_esperada": "actualizar_estado_ticket"},
]

filas = []
for caso in casos_prueba:
    plan = planificar(caso["consulta"])
    accion_obtenida = plan[0].nombre if plan else None
    filas.append({
        "consulta": caso["consulta"],
        "accion_esperada": caso["accion_esperada"],
        "accion_obtenida": accion_obtenida,
        "correcto": accion_obtenida == caso["accion_esperada"],
    })

pd.DataFrame(filas)

,consulta,accion_esperada,accion_obtenida,correcto
0,Buscar cliente Carla,buscar_cliente,buscar_cliente,True
1,Crear ticket para Bruno Díaz por duda de factu...,crear_ticket,crear_ticket,True
2,Muestra tickets abiertos,listar_tickets,listar_tickets,True
3,Dame un resumen de tickets,resumir_tickets,resumir_tickets,True
4,Cierra tic_002,actualizar_estado_ticket,actualizar_estado_ticket,True


Acciones como buscar clientes o listar tickets no modifican la base.

In [30]:
for consulta in [
    "Buscar cliente Ana",
    "Lista tickets abiertos",
    "Lista tickets abiertos de prioridad alta",
    "Dame un resumen general de tickets",
]:
    resultado = agente.responder(consulta)
    print("=" * 90)
    print("Usuario:", consulta)
    print(resultado["respuesta"])

Usuario: Buscar cliente Ana
Clientes encontrados:
- cli_001: Ana Torres (ana@empresa.cl), plan premium, riesgo medio
Usuario: Lista tickets abiertos
Tickets encontrados:
- tic_001 | cliente cli_001 | abierto | alta | No puede iniciar sesión
- tic_002 | cliente cli_002 | abierto | media | Consulta sobre facturación
Usuario: Lista tickets abiertos de prioridad alta
Tickets encontrados:
- tic_001 | cliente cli_001 | abierto | alta | No puede iniciar sesión
Usuario: Dame un resumen general de tickets
Resumen de tickets:
- abierto - alta: 1
- abierto - media: 1
- en_progreso - alta: 1


Crear ticket sí escribe en la base.

In [31]:
consulta = "Crear ticket para Ana Torres por error al entrar al panel de pagos prioridad alta"
resultado = agente.responder(consulta)
print("Usuario:", consulta)
print(resultado["respuesta"])
print("\nTraza:")
print(json.dumps(resultado["trazas"], ensure_ascii=False, indent=2))

Usuario: Crear ticket para Ana Torres por error al entrar al panel de pagos prioridad alta
Ticket creado: tic_004 para cli_001 | prioridad alta | estado abierto | asunto: Crear ticket para Ana Torres por error al entrar al panel de pagos prioridad alta

Traza:
[
  {
    "accion": "crear_ticket",
    "argumentos": {
      "cliente_id": "cli_001",
      "asunto": "Crear ticket para Ana Torres por error al entrar al panel de pagos prioridad alta",
      "prioridad": "alta"
    },
    "requiere_confirmacion": false,
    "confirmado": false,
    "resultado": {
      "ok": true,
      "mensaje": "Ticket creado correctamente.",
      "datos": {
        "id": "tic_004",
        "cliente_id": "cli_001",
        "asunto": "Crear ticket para Ana Torres por error al entrar al panel de pagos prioridad alta",
        "prioridad": "alta",
        "estado": "abierto",
        "creado_en": "2026-08-02T18:47:19.761710+00:00",
        "actualizado_en": "2026-08-02T18:47:19.761728+00:00"
      }
    }
  }

In [32]:
# Verificamos que el ticket quedó en la base activa.
pd.DataFrame(repo.listar_tickets())

,actualizado_en,id,prioridad,cliente_id,estado,asunto,creado_en
0,2026-08-02T18:08:57.762455+00:00,tic_001,alta,cli_001,abierto,No puede iniciar sesión,2026-08-02T18:08:57.762437+00:00
1,2026-08-02T18:08:58.117541+00:00,tic_002,media,cli_002,abierto,Consulta sobre facturación,2026-08-02T18:08:58.117522+00:00
2,2026-08-02T18:08:58.738583+00:00,tic_003,alta,cli_003,en_progreso,Error intermitente en dashboard,2026-08-02T18:08:58.738564+00:00
3,2026-08-02T18:47:19.761728+00:00,tic_004,alta,cli_001,abierto,Crear ticket para Ana Torres por error al entr...,2026-08-02T18:47:19.761710+00:00


Cambiar estado de un ticket requiere confirmación. Primero lo intentamos sin confirmar.

In [33]:
consulta = "Cierra tic_001"
resultado_sin_confirmar = agente.responder(consulta, confirmar=False)
print(resultado_sin_confirmar["respuesta"])
print("\nTraza:")
print(json.dumps(resultado_sin_confirmar["trazas"], ensure_ascii=False, indent=2))

No pude completar `actualizar_estado_ticket`: Se requiere confirmación humana para cambiar estado.. Datos: {'requiere_confirmacion': True}

Traza:
[
  {
    "accion": "actualizar_estado_ticket",
    "argumentos": {
      "ticket_id": "tic_001",
      "estado": "cerrado"
    },
    "requiere_confirmacion": true,
    "confirmado": false,
    "resultado": {
      "ok": false,
      "mensaje": "Se requiere confirmación humana para cambiar estado.",
      "datos": {
        "requiere_confirmacion": true
      }
    }
  }
]


Ahora repetimos con confirmación explícita.

In [34]:
resultado_confirmado = agente.responder(consulta, confirmar=True)
print(resultado_confirmado["respuesta"])
pd.DataFrame(repo.listar_tickets())

Ticket tic_001 actualizado a estado `cerrado`.


,actualizado_en,id,prioridad,cliente_id,asunto,estado,creado_en
0,2026-08-02T18:49:39.580987+00:00,tic_001,alta,cli_001,No puede iniciar sesión,cerrado,2026-08-02T18:08:57.762437+00:00
1,2026-08-02T18:08:58.117541+00:00,tic_002,media,cli_002,Consulta sobre facturación,abierto,2026-08-02T18:08:58.117522+00:00
2,2026-08-02T18:08:58.738583+00:00,tic_003,alta,cli_003,Error intermitente en dashboard,en_progreso,2026-08-02T18:08:58.738564+00:00
3,2026-08-02T18:47:19.761728+00:00,tic_004,alta,cli_001,Crear ticket para Ana Torres por error al entr...,abierto,2026-08-02T18:47:19.761710+00:00


Cada interacción queda registrada. En Firestore se guarda en `memoria_agente`.

In [35]:
memoria = repo.listar_memoria(limite=10)
pd.DataFrame(memoria)

,tipo,id,creado_en,modo_repo,contenido
0,interaccion_agente,mem_a97701df,2026-08-02T18:49:40.675928+00:00,firestore,"{'mensaje_usuario': 'Cierra tic_001', 'segundo..."
1,interaccion_agente,mem_8bcf1d7a,2026-08-02T18:48:26.043720+00:00,firestore,"{'mensaje_usuario': 'Cierra tic_001', 'segundo..."
2,interaccion_agente,mem_e8152e95,2026-08-02T18:47:20.191597+00:00,firestore,{'mensaje_usuario': 'Crear ticket para Ana Tor...
3,interaccion_agente,mem_a4e0846e,2026-08-02T18:45:55.871356+00:00,firestore,{'mensaje_usuario': 'Dame un resumen general d...
4,interaccion_agente,mem_f719a803,2026-08-02T18:45:54.847506+00:00,firestore,{'mensaje_usuario': 'Lista tickets abiertos de...
5,interaccion_agente,mem_2a49e9d9,2026-08-02T18:45:53.375278+00:00,firestore,"{'mensaje_usuario': 'Lista tickets abiertos', ..."
6,interaccion_agente,mem_54988322,2026-08-02T18:45:51.297562+00:00,firestore,"{'mensaje_usuario': 'Buscar cliente Ana', 'seg..."
7,interaccion_agente,mem_e77a25e5,2026-08-02T18:45:21.517779+00:00,firestore,{'mensaje_usuario': 'Dame un resumen general d...
8,interaccion_agente,mem_22245fca,2026-08-02T18:45:20.277292+00:00,firestore,{'mensaje_usuario': 'Lista tickets abiertos de...
9,interaccion_agente,mem_13ec2766,2026-08-02T18:45:19.013998+00:00,firestore,"{'mensaje_usuario': 'Lista tickets abiertos', ..."


In [36]:
# Inspecciona una memoria completa.
if memoria:
    print(json.dumps(memoria[0], ensure_ascii=False, indent=2))

{
  "tipo": "interaccion_agente",
  "id": "mem_a97701df",
  "creado_en": "2026-08-02T18:49:40.675928+00:00",
  "modo_repo": "firestore",
  "contenido": {
    "mensaje_usuario": "Cierra tic_001",
    "segundos": 2.231,
    "trazas": [
      {
        "resultado": {
          "datos": {
            "actualizado_en": "2026-08-02T18:49:39.580987+00:00",
            "id": "tic_001",
            "prioridad": "alta",
            "cliente_id": "cli_001",
            "estado": "cerrado",
            "asunto": "No puede iniciar sesión",
            "creado_en": "2026-08-02T18:08:57.762437+00:00"
          },
          "mensaje": "Estado actualizado correctamente.",
          "ok": true
        },
        "accion": "actualizar_estado_ticket",
        "requiere_confirmacion": true,
        "confirmado": true,
        "argumentos": {
          "estado": "cerrado",
          "ticket_id": "tic_001"
        }
      }
    ],
    "plan": [
      {
        "requiere_confirmacion": true,
        "argument

## Límites del planificador actual

Este agente funciona para un conjunto pequeño de frases. Un LLM planner o LangChain/LangGraph permitiría interpretar solicitudes más variadas, pero también exigiría más guardrails (esquemas).

Limitaciones actuales:

- No resuelve ambigüedad conversando.
- No maneja múltiples acciones complejas en una frase larga.
- No entiende todos los sinónimos.
- No tiene permisos por usuario.

## Propuesto: cómo llevarlo a un agente con LLM

Para conectar un LL, el diseño cambiaría de la siguiente manera:

```text
Mensaje -> LLM planner con lista de herramientas -> tool call -> observación -> LLM respuesta
```

La especificación de herramientas sería la misma:

- `buscar_cliente(texto)`
- `crear_ticket(cliente_id, asunto, prioridad)`
- `listar_tickets(estado, prioridad, cliente_id)`
- `actualizar_estado_ticket(ticket_id, estado, confirmado)`
- `resumir_tickets()`

In [ ]:
# Plantilla conceptual para usar estas herramientas con frameworks como LangChain.
# Antes deberás configurar el proveedor/modelo.

PLANTILLA_SYSTEM_PROMPT = """
Eres un agente de soporte. Puedes usar herramientas para consultar y modificar tickets.
Reglas:
- No inventes clientes ni tickets.
- Para cambios de estado, exige confirmación humana.
- Usa herramientas antes de afirmar datos operacionales.
- Resume las acciones realizadas al final.
Herramientas disponibles:
1. buscar_cliente(texto)
2. crear_ticket(cliente_id, asunto, prioridad)
3. listar_tickets(estado, prioridad, cliente_id)
4. actualizar_estado_ticket(ticket_id, estado, confirmado)
5. resumir_tickets()
""".strip()

print(PLANTILLA_SYSTEM_PROMPT)

### 1: planificador LLM con OpenRouter

Este bloque convierte lenguaje natural en un plan JSON. El LLM no ejecuta herramientas directamente: solo propone `accion`, `argumentos` y si requiere confirmación. Luego el ejecutor del notebook valida y llama las funciones.


In [ ]:
CATALOGO_HERRAMIENTAS_LLM = [
    {
        "nombre": "buscar_cliente",
        "argumentos": {"texto": "texto de búsqueda"},
        "descripcion": "Busca clientes por nombre, email, plan o identificador.",
    },
    {
        "nombre": "crear_ticket",
        "argumentos": {"cliente_id": "cli_001", "asunto": "texto", "prioridad": "baja|media|alta|critica"},
        "descripcion": "Crea un ticket nuevo para un cliente existente.",
    },
    {
        "nombre": "listar_tickets",
        "argumentos": {"estado": "abierto|en_progreso|cerrado|null", "prioridad": "baja|media|alta|critica|null", "cliente_id": "cli_001|null"},
        "descripcion": "Lista tickets filtrados.",
    },
    {
        "nombre": "actualizar_estado_ticket",
        "argumentos": {"ticket_id": "tic_001", "estado": "abierto|en_progreso|cerrado"},
        "descripcion": "Cambia el estado de un ticket. Requiere confirmación humana.",
    },
    {
        "nombre": "resumir_tickets",
        "argumentos": {},
        "descripcion": "Resume tickets por estado y prioridad.",
    },
]


def extraer_json_objeto(texto: str) -> Dict[str, Any]:
    """Extrae el primer objeto JSON de una respuesta del modelo."""
    texto = texto.strip()
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        inicio = texto.find("{")
        fin = texto.rfind("}")
        if inicio == -1 or fin == -1 or fin <= inicio:
            raise
        return json.loads(texto[inicio:fin + 1])


def planificar_con_openrouter(mensaje_usuario: str) -> List[Accion]:
    system = f"""
Eres un planificador de herramientas para un agente de soporte conectado a Firestore.
No respondas al usuario final. Devuelve solo JSON válido con esta forma:
{{"accion": "nombre_herramienta", "argumentos": {{}}, "requiere_confirmacion": false}}

Herramientas permitidas:
{json.dumps(CATALOGO_HERRAMIENTAS_LLM, ensure_ascii=False, indent=2)}

Reglas:
- Usa solo una herramienta por turno en este laboratorio.
- Si faltan datos para crear ticket, usa buscar_cliente.
- actualizar_estado_ticket siempre requiere confirmación.
- No inventes IDs si el usuario no los entrega o no se pueden inferir.
""".strip()

    texto = chat_openrouter([
        {"role": "system", "content": system},
        {"role": "user", "content": mensaje_usuario},
    ], temperatura=0.0, max_tokens=300)
    obj = extraer_json_objeto(texto)
    nombre = obj.get("accion", "ayuda")
    argumentos = obj.get("argumentos") or {}
    requiere = bool(obj.get("requiere_confirmacion", nombre == "actualizar_estado_ticket"))

    if nombre not in HERRAMIENTAS:
        return [Accion("ayuda", {"mensaje": mensaje_usuario})]
    return [Accion(nombre, argumentos, requiere_confirmacion=requiere)]


consulta_llm = "Cierra tic_002 porque el cliente confirmó que ya está resuelto"
if openrouter_disponible():
    plan_llm = planificar_con_openrouter(consulta_llm)
    print("Plan propuesto:", [a.__dict__ for a in plan_llm])
    for accion in plan_llm:
        resultado = ejecutar_accion(accion, confirmar=False)
        print(formatear_respuesta(accion, resultado))
else:
    print("Define OPENROUTER_API_KEY para probar el planificador LLM. El agente determinista sigue funcionando sin API externa.")


### 2: Agregar “demás herramientas”

El patrón usado para Firestore se repite para cualquier integración externa:

```text
1. Definir contrato de la herramienta
2. Validar entradas con esquema
3. Ejecutar integración externa
4. Devolver observación estructurada
5. Registrar memoria/traza
6. Exigir confirmación si hay efecto irreversible
```

Ejemplos de extensiones:

| Herramienta externa | Acción posible | Consideración |
|---|---|---|
| Email | Enviar resumen al cliente | Confirmación humana |
| Slack/Teams | Notificar ticket crítico | Evitar spam y filtrar canales |
| Calendar | Agendar reunión de soporte | Zona horaria y disponibilidad |
| Buscador/RAG | Recuperar documentación | Citar fuente y controlar inyección |

Abajo dejamos una plantilla de herramienta externa sin ejecución real.

In [ ]:
def herramienta_notificar_equipo(canal: str, mensaje: str, confirmado: bool = False) -> ToolResult:
    """Plantilla de integración externa: Slack, Teams, email u otro canal."""
    if not confirmado:
        return ToolResult(
            ok=False,
            mensaje="Se requiere confirmación humana antes de enviar notificaciones externas.",
            datos={"canal": canal, "mensaje_preparado": mensaje},
        )

    # Aquí iría la llamada, por ejemplo:
    # requests.post(WEBHOOK_URL, json={"text": mensaje})
    return ToolResult(
        ok=True,
        mensaje="Notificación simulada enviada.",
        datos={"canal": canal, "mensaje": mensaje},
    )

print(herramienta_notificar_equipo("soporte", "Ticket crítico creado", confirmado=False).model_dump())

## Referencias

- Firebase Admin SDK: https://firebase.google.com/docs/admin/setup
- Firestore con librerías de servidor: https://firebase.google.com/docs/firestore/quickstart-server
- LangChain Agents: https://docs.langchain.com/oss/python/langchain/agents
- ReAct: https://arxiv.org/abs/2210.03629
- Toolformer: https://arxiv.org/abs/2302.04761
- Survey de agentes LLM: https://arxiv.org/abs/2308.11432
- Generative Agents: https://arxiv.org/abs/2304.03442
- OpenRouter Quickstart: https://openrouter.ai/docs/quickstart
- OpenRouter API reference: https://openrouter.ai/docs/api/reference/overview
- OpenRouter Models API: https://openrouter.ai/docs/api/api-reference/models/get-models
